# Track B — Bước B2: Tách nhánh Khám phá (`DYDM` Discovery Branch + `NIW` Memory) trên `CUB-200`
- Nhánh: `duytung/04-cub-tach-nhanh-kham-pha`
- Tái sử dụng trực tiếp checkpoint `S0` (`ft_ckpt.pt`) và 5 chuỗi snapshot `colab_cub_ft_l25_r0_s0..s4` đã có sẵn trên Google Drive (không phải chạy lại 4 tiếng SVI!).
- Mở riêng **block Transformer cuối** của `DINOv1` + đầu phân loại cosine ở mỗi phiên $t=1..5$ với $\mathcal{L}_{\text{self-train}} + \mathcal{L}_{\text{GER}} + \mathcal{L}_{\text{KD}}$, giữ nguyên `Memory = NIW` cho lớp cũ.
- Tự động đo **CPU % peak/avg, RAM peak GB, GPU % peak, GPU VRAM peak MB** và thời gian từng seed `s0..s4`, xuất ra `cub_b2_summary.txt` / `cub_b2_summary.json`.

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / cập nhật repo lên ổ local /content (chạy git trong 2 giây)
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b duytung/04-cub-tach-nhanh-kham-pha https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin duytung/04-cub-tach-nhanh-kham-pha
!git checkout duytung/04-cub-tach-nhanh-kham-pha
!git reset --hard origin/duytung/04-cub-tach-nhanh-kham-pha
!git status -sb

In [ ]:
# 3. Cài gói thiếu, kiểm tra GPU và xác nhận dữ liệu CUB Phase 1 + s0..s4 trên Drive
!pip uninstall -y -q torchao
!pip install -q continuum numpyro prettytable

import glob, subprocess
has_gpu = subprocess.call(
    ['python', '-c', 'import jax; assert any(d.platform != "cpu" for d in jax.devices())']
) == 0
if not has_gpu:
    print('>>> JAX đang chạy CPU, thử cài jax[cuda12]...')
    !pip install -q -U "jax[cuda12]"

!python -c "import torch, jax, numpyro, continuum, prettytable, sklearn, scipy, datasets, peft; print('Imports OK! Torch CUDA:', torch.cuda.is_available(), '| JAX devices:', jax.devices())"

DRIVE_REPO = '/content/drive/MyDrive/VB_CGCD_Work/VB-CGCD-defect-testing-only'
ckpts = sorted(glob.glob(f'{DRIVE_REPO}/features/*__colab_cub_ft_l25_r0/ft_ckpt.pt'))
snaps = [s for s in range(5) if glob.glob(f'{DRIVE_REPO}/runs/*__colab_cub_ft_l25_r0_s{s}/snap/S5_14.pkl')]
print('>>> Tìm thấy ft_ckpt.pt:', ckpts[-1] if ckpts else 'KHÔNG CÓ')
print('>>> Tìm thấy snapshots CUB s0..s4:', snaps)
assert ckpts and len(snaps) == 5, 'Thiếu ft_ckpt.pt hoặc snapshots s0..s4 của CUB-200 trên Drive!'

In [ ]:
# 4. Chạy ngầm Bước B2 DYDM (STAGE='auto' | 's0'..'s4' | 'summary')
STAGE = 'auto'

import os, subprocess, time

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/cub_b2_bg.log'
PID_FILE = f'{WORK_DIR}/cub_b2_bg.pid'

running = False
if os.path.exists(PID_FILE):
    old_pid = open(PID_FILE).read().strip()
    if old_pid and os.path.exists(f'/proc/{old_pid}/status'):
        st = open(f'/proc/{old_pid}/status').read()
        if 'State:\tZ' not in st and 'State: Z' not in st:
            running = True
            print(f'>>> Pipeline đang chạy ngầm sẵn với PID {old_pid}. Chạy Cell 5 để xem log!')

if not running:
    proc = subprocess.Popen(
        ['nohup', 'bash', 'colab/run_cub_b2_bg.sh', STAGE],
        stdout=open(LOG_FILE, 'w'),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    with open(PID_FILE, 'w') as f:
        f.write(str(proc.pid))
    time.sleep(2)
    print(f'>>> Đã khởi chạy ngầm STAGE={STAGE}, PID={proc.pid}! Log: {LOG_FILE}')
    !head -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ, CPU/RAM/GPU peak từng seed & bảng kết quả Bước B2 (DYDM)
import os

WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/cub_b2_bg.log'
PID_FILE = f'{WORK_DIR}/cub_b2_bg.pid'
SUMMARY_OUT = f'{WORK_DIR}/cub_b2_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = 'State:\tZ' not in st and 'State: Z' not in st
print(f'Trạng thái PID={pid}:', 'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / XONG')

if os.path.exists(LOG_FILE):
    print('\n--- 40 dòng log gần nhất ---')
    !tail -n 40 {LOG_FILE}

if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG PEAK BƯỚC B2 (DYDM) ---')
    print(open(SUMMARY_OUT).read())